# Ditt Server

> The core abstraction for `Ditto` server.

In [ ]:
#| default_exp servers.ditto

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs
from fedai.optimizers.custom_optimizers import PerturbedGradientDescent

<torch._C.Generator>

## ServerDitto

In [ ]:
#| export
@AlgorithmRegistry.register_server("ditto")
class ServerDitto(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
                 

In [ ]:
#| export
@patch
def client_fn(self: ServerDitto, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = copy.deepcopy(self.model.state_dict())
        client_state['model_per'] = copy.deepcopy(self.model.state_dict())

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model

    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    optimizer = get_optimizer(self.cfg)(model.parameters(), **kwargs)
        
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer'] = optimizer

    model_per = create_model(self.cfg)
    model_per.load_state_dict(client_state['model_per'])
    client_state['model_per'] = model_per

    optimizer_per = PerturbedGradientDescent(model_per.parameters(), self.cfg.optimizer.lr, self.cfg.algorithm.mu)
    optimizer_per.load_state_dict(client_state['optimizer_per']) if 'optimizer_per' in client_state else None
    for state in optimizer_per.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer_per'] = optimizer_per

    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerDitto, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)

        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'model': self.model.state_dict(),
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
                    'model_per': self.state_mgr.get_state(id).get('model_per', None),
                    'optimizer_per': self.state_mgr.get_state(id).get('optimizer_per', None)
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()